# DS605 Lab 4: Model Training & Testing

Linear regression, train/test split, and all evaluation metrics implemented manually with NumPy only 


In [1]:
import numpy as np
import pandas as pd

df1 = pd.read_csv('df1.csv')

drop_cols = [c for c in ['name', 'price'] if c in df1.columns]
X = df1.drop(columns=drop_cols + ['log_price']).to_numpy(dtype=float)
y = df1['log_price'].to_numpy(dtype=float)

print("X shape:", X.shape, " y shape:", y.shape)


X shape: (48895, 48)  y shape: (48895,)


## 1. Train/Test Split

In [2]:
def train_test_split_manual(X, y, test_size=0.2, seed=42):
    """Shuffle row indices and slice them into train/test sets -- no sklearn."""
    n = X.shape[0]
    rng = np.random.default_rng(seed)
    shuffled_idx = rng.permutation(n)

    n_test = int(n * test_size)
    test_idx = shuffled_idx[:n_test]
    train_idx = shuffled_idx[n_test:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train, X_test, y_train, y_test = train_test_split_manual(X, y, test_size=0.2, seed=42)
print("Train size:", X_train.shape, " Test size:", X_test.shape)


Train size: (39116, 48)  Test size: (9779, 48)


## 2. Feature Scaling

Standardizing helps the linear regression solve stay numerically stable across features on very different scales (e.g. latitude vs. availability_365).

In [3]:
def standardize_fit(X):
    """Compute mean/std from TRAINING data only (avoids leaking test info)."""
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std[std == 0] = 1.0   # avoid divide-by-zero for constant columns
    return mean, std

def standardize_apply(X, mean, std):
    return (X - mean) / std


train_mean, train_std = standardize_fit(X_train)
X_train_scaled = standardize_apply(X_train, train_mean, train_std)
X_test_scaled = standardize_apply(X_test, train_mean, train_std)   # same mean/std as train, no leakage


## 3. Linear Regression(Gradient Descent)

In [4]:
def add_bias(X):
    """Add a column of 1s so the model can learn an intercept term."""
    return np.hstack([np.ones((X.shape[0], 1)), X])


def train_linear_regression(X, y, learning_rate=0.1, n_iterations=2000):
    """
    Multivariate linear regression trained with batch gradient descent.
    Model: y_hat = X @ weights
    Loss:  Mean Squared Error = mean((y_hat - y)^2)
    Update rule: weights -= learning_rate * gradient
    """
    X_b = add_bias(X)
    n_samples, n_features = X_b.shape
    weights = np.zeros(n_features)

    for i in range(n_iterations):
        y_pred = X_b @ weights
        error = y_pred - y

        # Gradient of MSE with respect to the weights
        gradient = (2 / n_samples) * (X_b.T @ error)
        weights -= learning_rate * gradient

    return weights


def predict_linear_regression(X, weights):
    X_b = add_bias(X)
    return X_b @ weights


weights = train_linear_regression(X_train_scaled, y_train, learning_rate=0.1, n_iterations=2000)
print("Trained weights shape:", weights.shape)


Trained weights shape: (49,)


## 4. Evaluation Metrics 

In [5]:
def mean_squared_error_manual(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def root_mean_squared_error_manual(y_true, y_pred):
    return np.sqrt(mean_squared_error_manual(y_true, y_pred))

def mean_absolute_error_manual(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)          # unexplained variance
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)  # total variance
    return 1 - (ss_res / ss_tot)


def expm1_manual(x):
    """Reverse of log1p: e^x - 1. Turns log_price predictions back into dollars."""
    return np.exp(x) - 1


train_pred = predict_linear_regression(X_train_scaled, weights)
test_pred = predict_linear_regression(X_test_scaled, weights)

train_r2 = r2_score_manual(y_train, train_pred)
test_r2 = r2_score_manual(y_test, test_pred)
test_rmse_log = root_mean_squared_error_manual(y_test, test_pred)

# Convert back to real dollar prices for an interpretable error.
test_pred_dollars = expm1_manual(test_pred)
test_actual_dollars = expm1_manual(y_test)
test_mae_dollars = mean_absolute_error_manual(test_actual_dollars, test_pred_dollars)
test_rmse_dollars = root_mean_squared_error_manual(test_actual_dollars, test_pred_dollars)

print(f"Train R2: {train_r2:.4f}")
print(f"Test R2:  {test_r2:.4f}")
print(f"Test RMSE (log price): {test_rmse_log:.4f}")
print(f"Test MAE ($):  {test_mae_dollars:.2f}")
print(f"Test RMSE ($): {test_rmse_dollars:.2f}")

gap = train_r2 - test_r2
print(f"\nTrain-Test R2 gap: {gap:.4f}", "(possible overfitting)" if gap > 0.15 else "(looks reasonable)")


Train R2: 0.5176
Test R2:  0.5320
Test RMSE (log price): 0.4708
Test MAE ($):  60.70
Test RMSE ($): 236.93

Train-Test R2 gap: -0.0144 (looks reasonable)


## 5. Save the Trained Model (weights + scaling parameters)

In [6]:
import pickle

model_artifact = {
    'weights': weights,
    'train_mean': train_mean,
    'train_std': train_std,
    'feature_names': list(df1.drop(columns=drop_cols + ['log_price']).columns),
}

with open('airbnb_price_model_scratch.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

print("Saved: airbnb_price_model_scratch.pkl")


Saved: airbnb_price_model_scratch.pkl
